# Test `consultation_summary`

This notebook includes:
- the function under test
- a sample `Visit` built from your Jane Smith example
- an offline mock-stream test that validates the `StreamingResponse` output
- an optional live API smoke test

In [ ]:
from datetime import date\n\nfrom fastapi.responses import StreamingResponse\nfrom openai import OpenAI\nfrom pydantic import BaseModel\n\n\nclass Visit(BaseModel):\n    patient_name: str\n    date: str\n    notes: str\n\n\nsystem_prompt = \"\"\"You are a concise clinical documentation assistant.\nSummarize the consultation clearly and safely.\"\"\"\n\n\ndef user_prompt_for(visit: Visit) -> str:\n    return f\"\"\"Patient Name: {visit.patient_name}\nDate: {visit.date}\nNotes:\n{visit.notes}\"\"\"\n\n\ndef consultation_summary(visit: Visit):\n    # Available for tracking/auditing\n    client = OpenAI()\n\n    user_prompt = user_prompt_for(visit)\n\n    prompt = [\n        {\"role\": \"system\", \"content\": system_prompt},\n        {\"role\": \"user\", \"content\": user_prompt},\n    ]\n\n    stream = client.chat.completions.create(\n        model=\"gpt-5-nano\",\n        messages=prompt,\n        stream=True,\n    )\n\n    def event_stream():\n        pending = \"\"\n        for chunk in stream:\n            text = chunk.choices[0].delta.content or \"\"\n            if not text:\n                continue\n\n            pending += text\n            lines = pending.split(\"\\n\")\n            pending = lines.pop()\n\n            for line in lines:\n                yield f\"data: {line}\\n\\n\"\n                yield \"data:  \\n\\n\"\n\n        if pending:\n            yield f\"data: {pending}\\n\\n\"\n\n    return StreamingResponse(event_stream(), media_type=\"text/event-stream\")

In [ ]:
sample_visit = Visit(\n    patient_name=\"Jane Smith\",\n    date=str(date.today()),\n    notes=(\n        \"Patient presents with persistent cough for 2 weeks. No fever.\\n\"\n        \"Chest clear on examination. Blood pressure 120/80.\\n\"\n        \"Likely viral bronchitis. Prescribed rest and fluids.\\n\"\n        \"Follow up if symptoms persist beyond another week.\"\n    ),\n)\n\nsample_visit

In [ ]:
from types import SimpleNamespace\n\n\nasync def collect_streaming_response(response: StreamingResponse) -> str:\n    parts = []\n    async for chunk in response.body_iterator:\n        if isinstance(chunk, bytes):\n            parts.append(chunk.decode(\"utf-8\"))\n        else:\n            parts.append(chunk)\n    return \"\".join(parts)\n\n\ndef make_chunk(text: str):\n    return SimpleNamespace(\n        choices=[SimpleNamespace(delta=SimpleNamespace(content=text))]\n    )\n\n\nfake_chunks = [\n    \"Consultation Summary\\n\",\n    \"Assessment: Likely viral bronchitis\\n\",\n    \"Plan: Rest, fluids, and follow up in one week if symptoms persist.\",\n]\n\n\nclass FakeOpenAI:\n    def __init__(self):\n        self.chat = SimpleNamespace(\n            completions=SimpleNamespace(create=self._create)\n        )\n\n    def _create(self, model, messages, stream):\n        assert model == \"gpt-5-nano\"\n        assert stream is True\n        assert \"Patient Name: Jane Smith\" in messages[1][\"content\"]\n        return iter(make_chunk(text) for text in fake_chunks)\n\n\noriginal_openai = OpenAI\nglobals()[\"OpenAI\"] = FakeOpenAI\n\ntry:\n    response = consultation_summary(sample_visit)\n    sse_output = await collect_streaming_response(response)\nfinally:\n    globals()[\"OpenAI\"] = original_openai\n\nexpected_sse_output = (\n    \"data: Consultation Summary\\n\\n\"\n    \"data:  \\n\\n\"\n    \"data: Assessment: Likely viral bronchitis\\n\\n\"\n    \"data:  \\n\\n\"\n    \"data: Plan: Rest, fluids, and follow up in one week if symptoms persist.\\n\\n\"\n)\n\nprint(sse_output)\n\nassert sse_output == expected_sse_output

In [ ]:
# Optional live smoke test. Run this only if OPENAI_API_KEY is set in your environment.\n#\n# live_response = consultation_summary(sample_visit)\n# print(await collect_streaming_response(live_response))